# Smart Scan Strategy for Electronic Warfare — Colab Pipeline

This notebook runs the tested model-only pipeline: configurable restricted download, leakage-safe whole-file splits, PDW preprocessing, RF replay simulation, open-loop and adaptive schedulers, optional DQN training, validation-only model selection, sealed test evaluation, and figures of merit.

**Security:** accept the gated Hugging Face dataset terms first. Enter the token only through the hidden prompt.  
**Test discipline:** leave `RUN_FINAL_TEST = False` until every scheduler/model choice is frozen.

In [ ]:
#@title Experiment size and controls
TRAIN_FILE_COUNT = 30 #@param {type:"integer"}
VALIDATION_FILE_COUNT = 10 #@param {type:"integer"}
TEST_FILE_COUNT = 10 #@param {type:"integer"}
TRAIN_START_ID = 0 #@param {type:"integer"}
VALIDATION_START_ID = 0 #@param {type:"integer"}
TEST_START_ID = 7 #@param {type:"integer"}
MODE = "stare" #@param ["stare", "scan"]

RUN_DQN = False #@param {type:"boolean"}
DQN_SEEDS = [41]  # Increase only after a timing smoke test.
DQN_EPOCHS = 1 #@param {type:"integer"}
RUN_FINAL_TEST = False #@param {type:"boolean"}

assert TRAIN_FILE_COUNT > 0 and VALIDATION_FILE_COUNT > 0 and TEST_FILE_COUNT > 0
assert DQN_EPOCHS > 0

## 1. Install the tested project

Upload `smart_scan_colab_project.zip`, generated with this notebook. The archive contains source code and configuration only—no local data, tokens, models, or results.

In [ ]:
from google.colab import files
from pathlib import Path
import os, shutil, subprocess, sys, zipfile

uploaded = files.upload()
archive_name = next((name for name in uploaded if name.endswith(".zip")), None)
if archive_name is None:
    raise ValueError("Upload smart_scan_colab_project.zip")

project_dir = Path("/content/smart_scan_ew")
if project_dir.exists():
    shutil.rmtree(project_dir)
project_dir.mkdir(parents=True)
with zipfile.ZipFile(archive_name) as archive:
    archive.extractall(project_dir)
expected_config = project_dir / "configs" / "scaled.yaml"
if not expected_config.is_file():
    raise FileNotFoundError(f"The uploaded bundle is missing {expected_config}")
os.chdir(project_dir)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[ml]"],
    check=True,
)
print("Project installed at", project_dir)

In [ ]:
import gc
import getpass
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from smart_scan.config import (
    dqn_kwargs, linucb_kwargs, load_config, preprocess_kwargs,
    receiver_config, reward_config,
)
from smart_scan.data.audit import audit_manifest
from smart_scan.data.download import download_manifest
from smart_scan.data.preprocess import preprocess_manifest
from smart_scan.data.split_integrity import validate_split_manifests
from smart_scan.evaluation.aggregate import aggregate_result_files, save_aggregate
from smart_scan.evaluation.fom import load_and_save_figures_of_merit
from smart_scan.evaluation.run import benchmark, save_results
from smart_scan.training import train_dqn

root_candidates = [Path.cwd(), Path("/content/smart_scan_ew")]
PROJECT_ROOT = next(
    (root for root in root_candidates if (root / "configs/scaled.yaml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot find configs/scaled.yaml. Re-run Section 1 and upload "
        "smart_scan_colab_project.zip before running this cell."
    )
os.chdir(PROJECT_ROOT)
print("Using project root:", PROJECT_ROOT)

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
MANIFEST_DIR = Path("data/manifests/colab")
OUTPUT_DIR = Path("outputs/colab")
MODEL_DIR = OUTPUT_DIR / "models"
for path in (RAW_DIR, PROCESSED_DIR, MANIFEST_DIR, OUTPUT_DIR, MODEL_DIR):
    path.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = Path("configs/scaled.yaml")
configuration = load_config(CONFIG_PATH)

parameter_defaults = {
    "TRAIN_FILE_COUNT": 30, "VALIDATION_FILE_COUNT": 10,
    "TEST_FILE_COUNT": 10, "TRAIN_START_ID": 0,
    "VALIDATION_START_ID": 0, "TEST_START_ID": 7,
    "MODE": "stare", "RUN_DQN": False,
    "DQN_SEEDS": [41], "DQN_EPOCHS": 1,
    "RUN_FINAL_TEST": False,
}
for parameter_name, default_value in parameter_defaults.items():
    if parameter_name not in globals():
        globals()[parameter_name] = default_value
print("Experiment parameters:", {name: globals()[name] for name in parameter_defaults})

## 2. Create and verify whole-file splits

Emitter IDs are local to a scenario. Complete HDF5 scenarios stay in exactly one official split; no windows or pulses are divided between train, validation, and test.

In [ ]:
DATASET_ID = "alan-turing-institute/turing-synthetic-radar-dataset"

def records(split, start, count):
    return [
        {"split": split, "mode": MODE, "config_id": config_id}
        for config_id in range(start, start + count)
    ]

development_manifest = {
    "dataset_id": DATASET_ID,
    "description": "Colab development split; whole files only.",
    "files": (
        records("train", TRAIN_START_ID, TRAIN_FILE_COUNT)
        + records("val", VALIDATION_START_ID, VALIDATION_FILE_COUNT)
    ),
}
test_manifest = {
    "dataset_id": DATASET_ID,
    "description": "Sealed Colab holdout; do not inspect before model selection.",
    "files": records("test", TEST_START_ID, TEST_FILE_COUNT),
}

development_manifest_path = MANIFEST_DIR / "development.json"
test_manifest_path = MANIFEST_DIR / "sealed_test.json"
development_manifest_path.write_text(json.dumps(development_manifest, indent=2))
test_manifest_path.write_text(json.dumps(test_manifest, indent=2))

integrity = validate_split_manifests(
    [development_manifest_path, test_manifest_path]
)
assert integrity["duplicate_source_files"] == 0
assert integrity["counts"] == {
    "train": TRAIN_FILE_COUNT,
    "val": VALIDATION_FILE_COUNT,
    "test": TEST_FILE_COUNT,
}
integrity

## 3. Securely download only train and validation data

The test manifest is deliberately not downloaded here.

In [ ]:
hf_token = getpass.getpass("Hugging Face read token (hidden): ")
try:
    development_receipt = download_manifest(
        development_manifest_path, RAW_DIR, token=hf_token
    )
finally:
    del hf_token
gc.collect()

print("Downloaded/verified", len(development_receipt), "development files")
pd.DataFrame(development_receipt)[
    ["split", "mode", "config_id", "size_bytes", "sha256"]
].head()

## 4. Audit and preprocess PDWs

The cache retains aggregate grids and exact pulse events. Dataset emitter labels are retained only as evaluator/reward truth. They are **not** present in scheduler contexts or DQN state vectors.

In [ ]:
development_audit_path = OUTPUT_DIR / "development_data_audit.json"
development_audit = audit_manifest(
    development_manifest_path, RAW_DIR, development_audit_path
)
processed_development = preprocess_manifest(
    development_manifest_path,
    RAW_DIR,
    PROCESSED_DIR,
    **preprocess_kwargs(configuration),
)
audit_frame = pd.DataFrame(development_audit)
display(audit_frame[
    ["split", "config_id", "pulses", "emitters", "duration_s",
     "frequency_min_mhz", "frequency_max_mhz",
     "amplitude_min_db", "amplitude_max_db"]
].describe(include="all"))
print("Processed episodes:", len(processed_development))

In [ ]:
train_paths = sorted(
    (PROCESSED_DIR / MODE / "train").glob("config_*.npz"),
    key=lambda p: int(p.stem.split("_")[-1]),
)[:TRAIN_FILE_COUNT]
validation_paths = sorted(
    (PROCESSED_DIR / MODE / "val").glob("config_*.npz"),
    key=lambda p: int(p.stem.split("_")[-1]),
)[:VALIDATION_FILE_COUNT]
assert len(train_paths) == TRAIN_FILE_COUNT
assert len(validation_paths) == VALIDATION_FILE_COUNT
print("Train:", len(train_paths), "Validation:", len(validation_paths))

## 5. Evaluate open-loop, periodic, and adaptive contextual-bandit schedulers

LinUCB starts cold and learns online inside each scenario. Its state is reset between scenarios.

In [ ]:
receiver = receiver_config(configuration)
reward = reward_config(configuration)
linucb_options = linucb_kwargs(configuration)
dqn_options = dqn_kwargs(configuration)

validation_result_paths = []
for episode_path in validation_paths:
    output_path = OUTPUT_DIR / "validation" / f"{episode_path.stem}.json"
    rows = benchmark(
        __import__("smart_scan.data.episode", fromlist=["Episode"]).Episode.load(episode_path),
        ["round_robin", "periodic", "linucb"],
        seeds=1,
        receiver=receiver,
        reward=reward,
        linucb_options=linucb_options,
    )
    save_results(rows, output_path)
    validation_result_paths.append(output_path)

validation_aggregate = aggregate_result_files(
    validation_result_paths, bootstrap_samples=10000, seed=42
)
validation_aggregate_path = save_aggregate(
    validation_aggregate, OUTPUT_DIR / "validation_aggregate.json"
)
validation_fom_path = load_and_save_figures_of_merit(
    validation_aggregate_path, OUTPUT_DIR / "validation_figures_of_merit.json"
)
pd.DataFrame(
    json.loads(validation_fom_path.read_text())["figures_of_merit"]
)

## 6. Optional compact DQN training

DQN is far more expensive than LinUCB. Start with one seed, measure runtime, and then increase seeds. Only training episodes enter the replay buffer.

In [ ]:
dqn_models = {}
if RUN_DQN:
    for seed in DQN_SEEDS:
        model_path = MODEL_DIR / f"dqn_adaptive_seed{seed}.pt"
        train_dqn(
            train_paths,
            epochs=DQN_EPOCHS,
            seed=seed,
            output_path=model_path,
            receiver=receiver,
            reward=reward,
            dqn_options=dqn_options,
        )
        dqn_models[seed] = model_path
    print("DQN models:", dqn_models)
else:
    print("DQN training skipped. Set RUN_DQN=True when desired.")

In [ ]:
dqn_validation = {}
if RUN_DQN:
    for seed, model_path in dqn_models.items():
        result_paths = []
        for episode_path in validation_paths:
            output_path = (
                OUTPUT_DIR / "validation_dqn" / f"seed{seed}_{episode_path.stem}.json"
            )
            rows = benchmark(
                __import__("smart_scan.data.episode", fromlist=["Episode"]).Episode.load(episode_path),
                ["dqn"],
                seeds=1,
                dqn_model=model_path,
                receiver=receiver,
                reward=reward,
                dqn_options=dqn_options,
            )
            save_results(rows, output_path)
            result_paths.append(output_path)
        aggregate = aggregate_result_files(result_paths, bootstrap_samples=10000, seed=42)
        aggregate_path = save_aggregate(
            aggregate, OUTPUT_DIR / f"validation_dqn_seed{seed}_aggregate.json"
        )
        dqn_validation[seed] = aggregate["summary"][0]
    selection = pd.DataFrame([
        {
            "seed": seed,
            "event_interception": summary["emitter_event_interception_ratio"]["mean"],
            "pulse_interception": summary["pulse_interception_ratio"]["mean"],
            "coverage": summary["unique_emitter_coverage"]["mean"],
            "first_intercept_delay_s": summary["average_first_intercept_delay_s"]["mean"],
            "switching": summary["average_switching_distance"]["mean"],
        }
        for seed, summary in dqn_validation.items()
    ]).sort_values(
        ["event_interception", "coverage", "first_intercept_delay_s"],
        ascending=[False, False, True],
    )
    display(selection)
    SELECTED_DQN_SEED = int(selection.iloc[0]["seed"])
    SELECTED_DQN_MODEL = dqn_models[SELECTED_DQN_SEED]
    print("Validation-selected DQN:", SELECTED_DQN_MODEL)

## 7. Freeze the decision, then optionally run the sealed test

Before changing `RUN_FINAL_TEST` to true, save the selected configuration and DQN seed. Do not tune anything after viewing test results.

In [ ]:
if RUN_FINAL_TEST:
    hf_token = getpass.getpass("Hugging Face read token for sealed test (hidden): ")
    try:
        test_receipt = download_manifest(test_manifest_path, RAW_DIR, token=hf_token)
    finally:
        del hf_token
    gc.collect()

    audit_manifest(
        test_manifest_path, RAW_DIR, OUTPUT_DIR / "sealed_test_data_audit.json"
    )
    preprocess_manifest(
        test_manifest_path,
        RAW_DIR,
        PROCESSED_DIR,
        **preprocess_kwargs(configuration),
    )
    test_paths = sorted(
        (PROCESSED_DIR / MODE / "test").glob("config_*.npz"),
        key=lambda p: int(p.stem.split("_")[-1]),
    )
    allowed_ids = set(range(TEST_START_ID, TEST_START_ID + TEST_FILE_COUNT))
    test_paths = [p for p in test_paths if int(p.stem.split("_")[-1]) in allowed_ids]
    assert len(test_paths) == TEST_FILE_COUNT

    scheduler_names = ["round_robin", "periodic", "linucb"]
    selected_model = globals().get("SELECTED_DQN_MODEL")
    if selected_model is not None:
        scheduler_names.append("dqn")

    test_results = []
    for episode_path in test_paths:
        output_path = OUTPUT_DIR / "sealed_test" / f"{episode_path.stem}.json"
        rows = benchmark(
            __import__("smart_scan.data.episode", fromlist=["Episode"]).Episode.load(episode_path),
            scheduler_names,
            seeds=1,
            dqn_model=selected_model,
            receiver=receiver,
            reward=reward,
            dqn_options=dqn_options,
            linucb_options=linucb_options,
        )
        save_results(rows, output_path)
        test_results.append(output_path)

    test_aggregate = aggregate_result_files(
        test_results, bootstrap_samples=10000, seed=42
    )
    test_aggregate_path = save_aggregate(
        test_aggregate, OUTPUT_DIR / "sealed_test_aggregate.json"
    )
    test_fom_path = load_and_save_figures_of_merit(
        test_aggregate_path, OUTPUT_DIR / "sealed_test_figures_of_merit.json"
    )
    display(pd.DataFrame(
        json.loads(test_fom_path.read_text())["figures_of_merit"]
    ))
else:
    print("Sealed test not touched. Freeze model selection before enabling this cell.")

## 8. Plot core scheduler outcomes

In [ ]:
summary_frame = pd.DataFrame([
    {
        "scheduler": row["scheduler"],
        "event_interception": row["emitter_event_interception_ratio"]["mean"],
        "pulse_interception": row["pulse_interception_ratio"]["mean"],
        "coverage": row["unique_emitter_coverage"]["mean"],
        "first_intercept_delay_s": row["average_first_intercept_delay_s"]["mean"],
        "average_reward": row["average_reward"]["mean"],
    }
    for row in validation_aggregate["summary"]
])
display(summary_frame)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
summary_frame.plot.bar(x="scheduler", y="event_interception", ax=axes[0], legend=False)
summary_frame.plot.bar(x="scheduler", y="coverage", ax=axes[1], legend=False)
summary_frame.plot.bar(x="scheduler", y="first_intercept_delay_s", ax=axes[2], legend=False)
axes[0].set_title("Emitter-event interception")
axes[1].set_title("Unique-emitter coverage")
axes[2].set_title("First-intercept delay")
for ax in axes:
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

## Interpretation rules

- Pd, Pfa, and sensitivity are simulated receiver figures because the dataset contains PDWs, not raw IQ/noise.
- High conditional next-pulse accuracy may be influenced by emitters remaining in one band; compare it with oracle-best-band accuracy.
- A scheduler should improve interception **and** meet coverage, discovery-delay, revisit, and switching constraints.
- More files improve diversity and confidence, but cannot correct a misaligned reward or policy constraint by themselves.